# Notebook 07 — Train Fraud Detector (IsolationForest) → Export to .pkl

Trains an Isolation Forest on transaction features for anomaly/fraud detection.

In [ ]:
import os, json, hashlib, time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import joblib

EXPORTS_DIR = Path(os.getenv('EXPORTS_DIR', r'C:\Users\MJ\Desktop\Agric\jupyter\exports\models'))
TX_CSV = Path(os.getenv('TRANSACTION_DATASET_CSV', r'C:\Users\MJ\Desktop\Agric\jupyter\data\transactions\synthetic_transactions.csv'))
MODEL_VERSION = os.getenv('MODEL_VERSION', 'v1')
ALLOW_SYNTHETIC_DATA = os.getenv('ALLOW_SYNTHETIC_DATA', 'false').lower() == 'true'
MIN_ROWS = int(os.getenv('MIN_TRANSACTION_ROWS', '100'))
CONTAMINATION = float(os.getenv('FRAUD_CONTAMINATION', '0.05'))

EXPORTS_DIR.mkdir(parents=True, exist_ok=True)

FRAUD_FEATURES = [
    'amount', 'quantity', 'price_per_kg', 'hour',
    'farmer_history', 'buyer_history', 'is_off_hour',
    'log_amount', 'log_quantity',
]

def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

print('Fraud detector configuration')
print('TX_CSV:', TX_CSV)
print('EXPORTS_DIR:', EXPORTS_DIR)
print('ALLOW_SYNTHETIC_DATA:', ALLOW_SYNTHETIC_DATA)

In [ ]:
base_required_columns = {'amount', 'quantity', 'price_per_kg', 'hour', 'farmer_history', 'buyer_history'}

if TX_CSV.exists():
    df = pd.read_csv(TX_CSV)
    dataset_source = str(TX_CSV)
else:
    if not ALLOW_SYNTHETIC_DATA:
        raise FileNotFoundError(f'Transaction CSV not found: {TX_CSV}. Set TRANSACTION_DATASET_CSV or ALLOW_SYNTHETIC_DATA=true for smoke tests only.')
    print('No transaction CSV found — generating synthetic data for smoke-testing only')
    rng = np.random.default_rng(42)
    n = 10000
    df = pd.DataFrame({
        'amount': rng.lognormal(5, 1.5, n),
        'quantity': rng.lognormal(3, 1.2, n),
        'price_per_kg': rng.uniform(0.05, 3.0, n),
        'hour': rng.integers(0, 24, n),
        'farmer_history': rng.integers(0, 100, n),
        'buyer_history': rng.integers(0, 100, n),
    })
    dataset_source = 'synthetic_smoke_test'

missing_cols = sorted(base_required_columns - set(df.columns))
if missing_cols:
    raise ValueError(f'Transaction dataset missing required columns: {missing_cols}')
if len(df) < MIN_ROWS:
    raise ValueError(f'Transaction dataset has {len(df)} rows, minimum required is {MIN_ROWS}')

if 'is_off_hour' not in df.columns:
    df['is_off_hour'] = ((df['hour'] < 6) | (df['hour'] > 22)).astype(int)

df['log_amount'] = np.log1p(df['amount'])
df['log_quantity'] = np.log1p(df['quantity'])
df = df.dropna(subset=FRAUD_FEATURES)
X = df[FRAUD_FEATURES].values.astype(np.float32)
print('Dataset source:', dataset_source)
print('Feature matrix shape:', X.shape)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

iso = IsolationForest(
    n_estimators=200,
    contamination=CONTAMINATION,
    max_features=len(FRAUD_FEATURES),
    random_state=42,
    n_jobs=-1,
)
iso.fit(X_scaled)

labels = iso.predict(X_scaled)
anomaly_pct = (labels == -1).mean()
print(f'Anomaly rate: {anomaly_pct:.2%} (target ~{CONTAMINATION:.2%})')

In [ ]:
model_path = EXPORTS_DIR / 'fraud_detector_v1.pkl'
scaler_path = EXPORTS_DIR / 'fraud_scaler_v1.pkl'

joblib.dump(iso, model_path, compress=3)
joblib.dump(scaler, scaler_path, compress=3)
print('Saved:', model_path, scaler_path)

meta = {
    'model': 'fraud_detector_v1',
    'version': MODEL_VERSION,
    'format': 'pkl',
    'source_notebook': '07_train_fraud_detector.ipynb',
    'dataset_source': dataset_source,
    'dataset_rows': int(len(df)),
    'features': FRAUD_FEATURES,
    'contamination': CONTAMINATION,
    'anomaly_rate': round(float(anomaly_pct), 4),
    'sha256': sha256_file(model_path),
    'scaler_sha256': sha256_file(scaler_path),
    'created_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
}
metadata_path = EXPORTS_DIR / 'fraud_detector_metadata.json'
metadata_path.write_text(json.dumps(meta, indent=2))
print('Metadata:', json.dumps(meta, indent=2))

In [ ]:
m2 = joblib.load(model_path)
s2 = joblib.load(scaler_path)
sample = np.array([[500.0, 100.0, 5.0, 14, 10, 5, 0, np.log1p(500), np.log1p(100)]], dtype=np.float32)
pred  = m2.predict(s2.transform(sample))[0]
score = m2.score_samples(s2.transform(sample))[0]
print(f'Smoke-test fraud prediction: label={pred}, score={score:.4f} — fraud_detector_v1.pkl ready for production.')